# Lab 3: A Conditional Generative Model for Images

**based on lab from MIT course ["Introduction to Flow Matching and Diffusion Models 2026"](https://diffusion.csail.mit.edu/2026/index.html)**

In [1]:
from typing import Optional

from matplotlib import pyplot as plt
import torch
from tqdm import tqdm


from torchvision import datasets, transforms
from torchvision.utils import make_grid
from einops import rearrange
from torch.utils.data import DataLoader

from models import VAE, DiffusionTransformerFlowModel
from trainers import Trainer, VAETrainer

# Part 1. Loading datasets

In [2]:
device = torch.device("cuda")

cifar = datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=transforms.Compose(
        [
            transforms.RandomHorizontalFlip(),
            # Converts PIL Image to [0, 1] torch.Tensor
            transforms.ToTensor(),
            transforms.Normalize(  # Scales pixel values to [-1, 1]
                mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]
            ),
        ]
    ),
)

cifar_test = datasets.CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=transforms.Compose(
        [
            transforms.RandomHorizontalFlip(),
            # Converts PIL Image to [0, 1] torch.Tensor
            transforms.ToTensor(),
            transforms.Normalize(  # Scales pixel values to [-1, 1]
                mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]
            ),
        ]
    ),
)


def cifar_reverse_transform(x):
    x = (x + 1.0) / 2.0
    return torch.clamp(x, 0.0, 1.0)

## Part 2: Training process

### 2.1: Training a VAE

In [3]:
vae = VAE(
    data_channels=3,
    hidden_channels=[8, 16, 16],
    beta=0.01,
).to(device)

In [4]:
train_dataloader = DataLoader(dataset=cifar, batch_size=64)
test_dataloader = DataLoader(dataset=cifar_test, batch_size=64)

In [5]:
trainer = VAETrainer(
    dataloader=train_dataloader,
    reverse_transform=cifar_reverse_transform,
)

In [13]:
# trainer.train(
#     model=vae,
#     num_epochs=20,
#     lr=1e-3,
#     warmup_steps=500,
#     ckpt_every=5,
#     run_name="vae",
# )

In [14]:
trainer.load(model=vae, run_name="vae", resume_from=f"epoch_{20}_state")

Loading model with size: 0.158 MiB


In [15]:
@torch.no_grad()
def visualize_latent_interpolation(
    x1: torch.Tensor,
    x2: torch.Tensor,
    vae: VAE,
    n_steps: int,
    save_path: Optional[str] = None,
    reverse_transform=None,
):

    z1_mean, z1_logvar = vae.encode(x1)
    z1 = z1_mean + torch.exp(0.5 * z1_logvar) * torch.randn_like(z1_mean)  # 1 c h w

    z2_mean, z2_logvar = vae.encode(x2)
    z2 = z2_mean + torch.exp(0.5 * z2_logvar) * torch.randn_like(z2_mean)  # 1 c h w

    lambdas = torch.linspace(0, 1, n_steps).to(z1.device)
    zs = (1 - lambdas) * z1.unsqueeze(-1) + lambdas * z2.unsqueeze(
        -1
    )  # 1 c h w n_steps
    zs = rearrange(zs, "1 c h w n -> n c h w")
    samples = vae.decode(zs)  # n_steps 1 h w

    if reverse_transform is not None:
        samples = reverse_transform(samples)

    grid = make_grid(samples, nrow=n_steps, normalize=True, value_range=(0, 1))
    plt.figure(figsize=(n_steps * 2, 2))
    plt.imshow(grid.permute(1, 2, 0).cpu())
    plt.axis("off")
    plt.title("Latent Interpolation")
    if save_path is not None:
        plt.savefig(save_path)
        plt.close()
    else:
        plt.show()
    return samples

In [16]:
# Perform interpolation in the latent space
vae.eval()
samples, _ = next(iter(test_dataloader))[:2]
interpolated_samples = visualize_latent_interpolation(
    x1=samples[:1],
    x2=samples[1:2],
    vae=vae,
    n_steps=10,
    reverse_transform=cifar_reverse_transform,
)  # n_steps 3 h w

RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cpu and cuda:0! (when checking argument for argument weight in method wrapper_CUDA___slow_conv2d_forward)

In [ ]:
@torch.no_grad()
def estimate_latent_stats(
    vae: VAE, dataloader: DataLoader, batches=100, batch_size=128
):
    zs = []
    vae.eval()

    batch_num = 0
    for batch in tqdm(dataloader):
        x, _ = batch
        z_mean, _ = vae.encode(x)
        zs.append(z_mean)
        batch_num += 1
        if batch_num == 100:
            break

    z = torch.cat(zs, dim=0)
    mean = z.mean(dim=(0, 2, 3), keepdim=True)
    std = z.std(dim=(0, 2, 3), keepdim=True).clamp_min(1e-6)
    return mean, std


latent_mean, latent_std = estimate_latent_stats(vae, train_dataloader)

### 5.3 Building latent space dataset

In [ ]:
# @torch.no_grad()
# def convert_to_latent_dataset(
#     vae: VAE,
#     dataset,
#     batch_size: int = 256,
#     latent_stats=None,
# ):
#     device = next(vae.parameters()).device
#     was_training = vae.training
#     vae.eval()

#     loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
#     latents = []
#     labels = []

#     for x, y in tqdm(loader, desc="Converting dataset to latents"):
#         x = x.to(device)
#         z_mean, _ = vae.encode(x)

#         latents.append(z_mean.cpu())
#         labels.append(y.cpu())

#     if was_training:
#         vae.train()

#     return TensorDataset(torch.cat(latents), torch.cat(labels))

In [ ]:
# latent_cifar = convert_to_latent_dataset(vae, cifar)
# latent_datasampler = DataSampler(dataset=latent_cifar).to(device)

### 5.2 Training a Latent Diffusion Model

In [ ]:
# class LatentCFGTrainer(Trainer):
#     def __init__(
#         self,
#         datasampler: DataSampler,
#         vae: VAE,
#         path: GaussianConditionalProbabilityPath,
#         null_label: int,
#         eta: float = 0.0,
#         eps: float = 0.001,
#         latent_stats: tuple[float, float] = (0.0, 1.0),
#         reverse_transform=None,
#         **kwargs,
#     ):
#         assert eta > 0 and eta < 1
#         super().__init__(**kwargs)
#         self.datasampler = datasampler
#         self.vae = vae
#         self.path = path
#         self.eta = eta
#         self.eps = eps
#         self.path = path
#         self.null_label = null_label
#         self.latent_mean, self.latent_std = latent_stats
#         self.reverse_transform = reverse_transform

#         self.vae.eval()
#         for p in self.vae.parameters():
#             p.requires_grad_(False)

#     @torch.no_grad()
#     def sample(
#         self,
#         num_samples: int,
#         out_dir,
#         batch_size: int = 256,
#         guidance_scale: float = 1.5,
#         num_timesteps: int = 250,
#         use_raw=False,
#         seed=None,
#         overwrite=False,
#     ):
#         if seed is not None:
#             torch.manual_seed(seed)

#         out_path = Path(out_dir)
#         out_path.mkdir(parents=True, exist_ok=True)

#         existing = sorted(out_path.glob("*.png"))

#         if overwrite:
#             for path in existing:
#                 path.unlink()
#             existing = []
#         elif existing:
#             raise FileExistsError(
#                 f"{out_path} already contains PNG files. Use overwrite=True or a new directory."
#             )

#         sample_model = self.model if use_raw else self.ema_model
#         sample_model.eval()
#         device = next(sample_model.parameters()).device
#         ode = CFGVectorFieldODE(
#             sample_model,
#             guidance_scale=guidance_scale,
#             null_label=self.null_label,
#         )
#         simulator = EulerSimulator(ode)

#         written = 0
#         pbar = tqdm(total=num_samples, desc=f"export FID samples w={guidance_scale}")
#         while written < num_samples:
#             # Sample initial conditions
#             cur_bs = min(batch_size, num_samples - written)
#             y = (torch.arange(written, written + cur_bs, device=device) % 10).long()
#             z0 = self.path.p_simple.sample(cur_bs).to(device)

#             # Simulate
#             ts = (
#                 torch.linspace(0, 0.999, num_timesteps, device=device)
#                 .view(1, -1, 1, 1, 1)
#                 .expand(cur_bs, -1, 1, 1, 1)
#             )
#             z1 = simulator.simulate(z0, ts, y=y, use_tqdm=False)

#             # Decode
#             z1 = z1 * self.latent_std.to(device) + self.latent_mean.to(device)
#             x1 = self.vae.decode(z1)
#             if self.reverse_transform is not None:
#                 x1 = self.reverse_transform(x1)
#             x1 = torch.clamp(x1, 0.0, 1.0)

#             # Save to out_dir
#             x_uint8 = (
#                 (x1 * 255.0).round().to(torch.uint8).permute(0, 2, 3, 1).cpu().numpy()
#             )
#             for img in x_uint8:
#                 Image.fromarray(img).save(out_path / f"{written:06d}.png")
#                 written += 1

#             pbar.update(cur_bs)
#         pbar.close()

#     @torch.no_grad()
#     def visualize_samples(
#         self,
#         save_path: str = None,
#         samples_per_class: int = 5,
#         num_timesteps: int = 100,
#         guidance_scales: List[float] = [1.0, 1.5, 2.0],
#         use_tqdm=False,
#         use_raw=False,
#         title=None,
#     ):
#         # Graph
#         fig, axes = plt.subplots(
#             1, len(guidance_scales), figsize=(10 * len(guidance_scales), 10)
#         )

#         sample_model = self.model if use_raw else self.ema_model
#         sample_model.eval()
#         device = next(sample_model.parameters()).device

#         for idx, w in enumerate(guidance_scales):
#             ode = CFGVectorFieldODE(
#                 sample_model,
#                 guidance_scale=w,
#                 null_label=self.null_label,
#             )
#             simulator = EulerSimulator(ode)

#             # Sample initial conditions
#             y = (
#                 torch.tensor(list(range(10)), dtype=torch.int64)
#                 .repeat_interleave(samples_per_class)
#                 .to(device)
#             )
#             num_samples = y.shape[0]
#             z0 = self.path.p_simple.sample(num_samples).to(device)

#             # Simulate
#             ts = (
#                 torch.linspace(0, 0.999, num_timesteps)
#                 .view(1, -1, 1, 1, 1)
#                 .expand(num_samples, -1, 1, 1, 1)
#                 .to(device)
#             )
#             z1 = simulator.simulate(z0, ts, y=y, use_tqdm=use_tqdm)

#             # Decode
#             z1 = z1 * self.latent_std.to(device) + self.latent_mean.to(device)
#             x1 = self.vae.decode(z1)
#             if self.reverse_transform is not None:
#                 x1 = self.reverse_transform(x1)
#             x1 = torch.clamp(x1, 0.0, 1.0)

#             # Plot
#             grid = make_grid(x1, nrow=samples_per_class, normalize=False)
#             axes[idx].imshow(grid.permute(1, 2, 0).cpu())
#             axes[idx].axis("off")
#             axes[idx].set_title(f"Guidance: $w={w:.1f}$", fontsize=25)

#         # Save
#         if save_path is not None:
#             plt.savefig(save_path)
#             plt.close()
#         else:
#             plt.show()

#     def checkpoint(
#         self,
#         step: int,
#     ):
#         # Save model
#         torch.save(
#             self.model.state_dict(),
#             os.path.join(self.output_dir, f"step_{step:06d}_model.pt"),
#         )
#         torch.save(
#             self.ema_model.state_dict(),
#             os.path.join(self.output_dir, f"step_{step:06d}_ema_model.pt"),
#         )
#         torch.save(
#             self.opt.state_dict(),
#             os.path.join(self.output_dir, f"step_{step:06d}_opt.pt"),
#         )
#         torch.save(
#             {
#                 "steps": self.steps,
#                 "losses": self.losses,
#                 "losses_ema": self.losses_ema,
#             },
#             os.path.join(self.output_dir, f"step_{step:06d}_losses.pt"),
#         )

#         plt.figure()
#         plt.plot(self.steps, self.losses_ema)
#         plt.xlabel("Step")
#         plt.ylabel("EMA Loss")
#         plt.title("EMA Loss vs. Step")
#         plt.savefig(os.path.join(self.output_dir, f"step_{step:06d}_lossplot.png"))
#         plt.close()

#         # Save output visualization
#         self.visualize_samples(
#             save_path=os.path.join(self.output_dir, f"step_{step:06d}_output.png"),
#             title=f"Step {step}, loss={self.losses_ema[-1]:.4f}",
#         )

#     def get_train_loss(self, batch_size: int) -> torch.Tensor:
#         # Step 1: Sample z, y from data + encode
#         with torch.no_grad():
#             z, y = self.datasampler.sample(batch_size)
#             z_enc = (z - self.latent_mean) / self.latent_std

#         # Step 2: Set each label to 10 (i.e., null) with probability eta
#         mask = torch.rand(batch_size, device=y.device) < self.eta
#         y = torch.where(mask, self.null_label, y)

#         # Step 3: Sample t and x
#         t = torch.rand(batch_size, device=z_enc.device) * (1 - self.eps)
#         x = self.path.sample_conditional_path(z_enc, t)

#         u_target = self.path.conditional_vector_field(x, z_enc, t)
#         u_theta = self.model(x, t, y)

#         return torch.nn.functional.mse_loss(u_theta, u_target)

In [ ]:
# # Finally, let's train!

# vae = vae.to(device)

# # Initialize latent probability path
# c = 16
# img_size = 8

# path = GaussianConditionalProbabilityPath(
#     p_data=None,
#     p_simple_shape=[c, img_size, img_size],
#     alpha=LinearAlpha(),
#     beta=LinearBeta(),
# ).to(device)

# # Initialize model
# dit = DiffusionTransformerFlowModel(
#     img_size=img_size,
#     patch_size=1,
#     num_layers=12,
#     c=c,
#     dim=384,
#     heads=8,
#     final_dim=64,
#     n_classes=11,
# ).to(device)

# # Initialize trainer
# trainer = LatentCFGTrainer(
#     datasampler=latent_datasampler,
#     vae=vae,
#     path=path,
#     eta=0.1,
#     null_label=10,
#     latent_stats=(latent_mean, latent_std),
#     reverse_transform=cifar_reverse_transform,
# )

In [ ]:
# trainer.train(
#     model=dit,
#     num_steps=250_000,
#     lr=1e-4,
#     batch_size=256,
#     ckpt_every=25_000,
#     run_name="dit",
#     warmup_steps=5_000,
#     resume_step=0,
#     ema_decay=0.9999,
# )

In [ ]:
# trainer.train(
#     model=dit,
#     num_steps=200_000,
#     lr=1e-4,
#     batch_size=256,
#     ckpt_every=25_000,
#     run_name="dit",
#     warmup_steps=0,
#     resume_step=50_000,
#     ema_decay=0.9999,
# )

In [ ]:
# trainer.load(dit, "dit", 250_000)

## Part 6: Evaluation

In [ ]:
# def fid_guidance_sweep(
#     trainer,
#     root_dir,
#     guidance_scales=(1.0, 1.25, 1.5, 1.75, 2.0),
#     num_images=10_000,
#     batch_size=256,
#     num_timesteps=100,
#     split="train",
#     use_raw=False,
#     fid_batch_size=64,
#     fid_device=None,
# ):
#     """Export samples and compute FID for a quick guidance-scale sweep."""
#     scores = {}
#     for scale in guidance_scales:
#         out_dir = Path(root_dir) / f"w_{scale:g}"
#         try:
#             trainer.sample(
#                 num_samples=num_images,
#                 out_dir=out_dir,
#                 batch_size=batch_size,
#                 guidance_scale=scale,
#                 num_timesteps=num_timesteps,
#                 use_raw=use_raw,
#                 overwrite=False,
#             )
#         except:
#             print("already sampled")

#         scores[scale] = fid.compute_fid(
#             str(out_dir),
#             dataset_name="cifar10",
#             dataset_res=32,
#             dataset_split=split,
#             mode="clean",
#             batch_size=fid_batch_size,
#             device=fid_device or next(trainer.model.parameters()).device,
#             use_dataparallel=False,
#         )
#     return scores

In [ ]:
# fid_guidance_sweep(trainer, "samples/ema/")

In [ ]:
# # Raw comparison needs a fresh kernel and raw-only load to avoid holding both models:
# fid_guidance_sweep(trainer, "samples/raw/", use_raw=True)